In [ ]:
!pip install -q tifffile pillow joblib efficientnet_pytorch==0.7.1


In [ ]:
import glob
import os
import shutil
import subprocess
import sys

REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'main'
REPO_DIR = '/kaggle/working/repo'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, REPO_DIR], check=True)

PYTHON = sys.executable
subprocess.run(
    [
        PYTHON, '-m', 'pip', 'install', '-q',
        'tifffile', 'pillow', 'joblib', 'efficientnet_pytorch==0.7.1',
    ],
    check=True,
)
FOLDS_CSV = os.path.join(REPO_DIR, 'data', 'train_folds.csv')

def find_slide_dir():
    direct_candidates = [
        '/kaggle/input/prostate-cancer-grade-assessment/train_images',
        '/kaggle/input/competitions/prostate-cancer-grade-assessment/train_images',
    ]
    recursive_candidates = sorted(set(glob.glob('/kaggle/input/**/train_images', recursive=True)))
    candidates = []
    seen = set()
    for path in direct_candidates + recursive_candidates:
        if path not in seen:
            candidates.append(path)
            seen.add(path)

    png_fallback = None
    for path in candidates:
        if not os.path.isdir(path):
            continue
        has_raw_tiffs = bool(glob.glob(os.path.join(path, '*.tif'))) or bool(glob.glob(os.path.join(path, '*.tiff')))
        if has_raw_tiffs:
            return path, 'raw_tiff'
        has_pngs = bool(glob.glob(os.path.join(path, '*.png')))
        if has_pngs and png_fallback is None:
            png_fallback = path

    if png_fallback is not None:
        return png_fallback, 'png_fallback'
    return None, None

SLIDES_DIR, SLIDE_SOURCE = find_slide_dir()
if SLIDES_DIR is None:
    input_roots = sorted(glob.glob('/kaggle/input/*'))
    raise RuntimeError(
        'Could not find a PANDA train_images directory with .tif/.tiff or .png slides. '
        'Attach the PANDA raw slide dataset, competition data, or the resized PNG dataset. '
        f'Visible /kaggle/input entries: {input_roots[:20]}'
    )

N_TILES = 36
TILE_SIZE = 192
TILE_FORMAT = 'png'
OUT_DIR = f'/kaggle/working/panda_tiles_{N_TILES}x{TILE_SIZE}_{TILE_FORMAT}'
TRAIN_FOLD = 0
TRAIN_BACKBONE = 'efficientnet-b0'
TRAIN_LOSS = 'ordinal'
TRAIN_EPOCHS = 6
TRAIN_BATCH_SIZE = 2
TRAIN_NUM_WORKERS = 0
TRAIN_TILE_DIR = OUT_DIR
TRAIN_FEATURE_TAG = f'tiles{N_TILES}_imsize{TILE_SIZE}'
EXPECTED_WEIGHT = (
    f"{TRAIN_BACKBONE.replace('-', '')}_{TRAIN_FEATURE_TAG}_{TRAIN_LOSS}_fold{TRAIN_FOLD}.pth"
)
print('REPO_DIR    =', REPO_DIR)
print('SLIDES_DIR  =', SLIDES_DIR)
print('SLIDE_SOURCE=', SLIDE_SOURCE)
print('OUT_DIR     =', OUT_DIR)
print('FOLDS_CSV   =', FOLDS_CSV)
print('N_TILES     =', N_TILES)
print('TILE_SIZE   =', TILE_SIZE)
print('TILE_FORMAT =', TILE_FORMAT)
print('TRAIN_FOLD  =', TRAIN_FOLD)
print('EXPECTED_WEIGHT =', EXPECTED_WEIGHT)
working_free_gib = shutil.disk_usage('/kaggle/working').free / (1024 ** 3)
print('WORKING_FREE_GIB =', f'{working_free_gib:.1f}')
if SLIDE_SOURCE == 'png_fallback':
    print('WARNING: using resized PNGs as a low-resolution fallback to unblock tile-pipeline testing.')
stale_npy_dir = '/kaggle/working/panda_tiles_36x192'
if TILE_FORMAT != 'npy' and os.path.exists(stale_npy_dir):
    print('WARNING: stale NPY output directory detected:', stale_npy_dir)
    print('Remove it before a full run with: !rm -rf /kaggle/working/panda_tiles_36x192')


In [ ]:
#smoke test

import subprocess

cmd = [
    PYTHON, 'scripts/preprocess_tiles.py',
    '--slides-dir', SLIDES_DIR,
    '--output-dir', OUT_DIR,
    '--folds-csv', FOLDS_CSV,
    '--tile-size', str(TILE_SIZE),
    '--n-tiles', str(N_TILES),
    '--level', '1',
    '--format', TILE_FORMAT,
    '--limit', '5',
    '--n-jobs', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# full process

import subprocess

cmd = [
    PYTHON, 'scripts/preprocess_tiles.py',
    '--slides-dir', SLIDES_DIR,
    '--output-dir', OUT_DIR,
    '--folds-csv', FOLDS_CSV,
    '--tile-size', str(TILE_SIZE),
    '--n-tiles', str(N_TILES),
    '--level', '1',
    '--format', TILE_FORMAT,
    '--n-jobs', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
import os
import subprocess

if not os.path.isdir(TRAIN_TILE_DIR):
    raise RuntimeError(f'Tile directory does not exist: {TRAIN_TILE_DIR}')

tile_files = [
    name for name in os.listdir(TRAIN_TILE_DIR)
    if name.lower().endswith(('.png', '.npy', '.npz'))
]
if not tile_files:
    raise RuntimeError(f'No tile artifacts found in {TRAIN_TILE_DIR}')

cmd = [
    PYTHON, '-m', 'src.train',
    '--fold', str(TRAIN_FOLD),
    '--folds-csv', FOLDS_CSV,
    '--tile-dir', TRAIN_TILE_DIR,
    '--backbone', TRAIN_BACKBONE,
    '--loss', TRAIN_LOSS,
    '--n-tiles', str(N_TILES),
    '--tile-size', str(TILE_SIZE),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--feature-tag', TRAIN_FEATURE_TAG,
    '--amp',
    '--no-pin-memory',
    '--output-dir', '/kaggle/working',
]
print('Training tile dir:', TRAIN_TILE_DIR)
print('Tile artifact count:', len(tile_files))
print('Expected weight:', EXPECTED_WEIGHT)
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
